# 04 — Competitor & Opportunity Analysis

This notebook combines **customer demand clusters** with the actual competitor-store locations for **Blinkit, Zepto, and Instamart**.

**Goal:** identify geographic areas with relatively high demand and lower competitor coverage.

The opportunity screen is intentionally simple and is used as a decision-support layer, not as a claim of actual store deployment.

In [ ]:
import pandas as pd
import numpy as np
import os

from sklearn.cluster import KMeans

ORDERS_PATH = "../data/customer_orders.csv"
CLUSTERED_ORDERS_PATH = "../outputs/customer_orders_with_clusters.csv"
COMPETITOR_PATH = "../data/Competitors.csv"

orders = pd.read_csv(ORDERS_PATH)
competitors = pd.read_csv(COMPETITOR_PATH)

# Reuse the cluster assignments generated by Notebook 03 when available.
try:
    clustered_orders = pd.read_csv(CLUSTERED_ORDERS_PATH)
except FileNotFoundError:
    print("Clustered order file not found. Creating K=5 clusters here as a fallback.")
    coords = orders[["Customer_Lat", "Customer_Lon"]].dropna()
    km = KMeans(n_clusters=5, random_state=42, n_init=20)
    labels = km.fit_predict(coords)
    orders.loc[coords.index, "cluster"] = labels + 1
    clustered_orders = orders
    os.makedirs("../outputs", exist_ok=True)

clustered_orders.head()

In [ ]:
# Keep only Bengaluru competitor records.
bengaluru_comp = competitors[
    competitors["City"].astype(str).str.strip().str.lower().eq("bengaluru")
    | competitors["City"].astype(str).str.strip().str.lower().eq("bangalore")
].copy()

print("Customer orders:", len(clustered_orders))
print("Bengaluru competitor stores:", len(bengaluru_comp))
print("\nCompetitors by company:")
print(bengaluru_comp["Company"].value_counts())

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0088
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

## Create Small Geographic Demand Cells

A small geographic grid is used to summarize local customer demand. This is not a predefined Bengaluru zoning system; it is only a geographic aggregation layer for comparing demand and competition.

In [ ]:
# Approximate grid size: about 1 km in latitude/longitude.
cell_size = 0.009

x = clustered_orders.dropna(subset=["Customer_Lat", "Customer_Lon"]).copy()
x["grid_lat_id"] = np.floor((x["Customer_Lat"] - x["Customer_Lat"].min()) / cell_size).astype(int)
x["grid_lon_id"] = np.floor((x["Customer_Lon"] - x["Customer_Lon"].min()) / cell_size).astype(int)

grid = (
    x.groupby(["grid_lat_id", "grid_lon_id"], as_index=False)
     .agg(
         total_orders=("Order_ID", "count"),
         area_lat=("Customer_Lat", "mean"),
         area_lon=("Customer_Lon", "mean"),
         cluster=("cluster", lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan)
     )
)

grid.head()

In [ ]:
# Compute competitor count within 2 km and nearest competitor distance.
comp_lat = bengaluru_comp["Latitude"].to_numpy()
comp_lon = bengaluru_comp["Longitude"].to_numpy()

competitor_counts = []
nearest_distances = []

for lat, lon in zip(grid["area_lat"], grid["area_lon"]):
    d = haversine_km(lat, lon, comp_lat, comp_lon)
    competitor_counts.append(int((d <= 2.0).sum()))
    nearest_distances.append(float(d.min()) if len(d) else np.nan)

grid["competitors_within_2km"] = competitor_counts
grid["nearest_competitor_km"] = nearest_distances

grid[["total_orders", "competitors_within_2km", "nearest_competitor_km"]].describe()

## Demand and Competition Classification

For this case study:

- **High demand:** top demand tier of the observed grid cells.
- **Lower competition:** relatively fewer competitor stores within 2 km.
- **Opportunity:** combines the two signals.

The thresholds below are transparent case-study rules, not industry standards.

In [ ]:
demand_high_threshold = grid["total_orders"].quantile(0.75)
demand_medium_threshold = grid["total_orders"].quantile(0.40)

grid["demand_level"] = np.select(
    [
        grid["total_orders"] >= demand_high_threshold,
        grid["total_orders"] >= demand_medium_threshold
    ],
    ["High Demand", "Medium Demand"],
    default="Low Demand"
)

grid["competition_level"] = np.select(
    [
        grid["competitors_within_2km"] <= 2,
        grid["competitors_within_2km"] <= 5
    ],
    ["Low Competition", "Medium Competition"],
    default="High Competition"
)

def classify_opportunity(row):
    if row["demand_level"] == "High Demand" and row["competitors_within_2km"] <= 2:
        return "High Opportunity"
    if row["demand_level"] in ["High Demand", "Medium Demand"] and row["competitors_within_2km"] <= 5:
        return "Medium Opportunity"
    return "Low Opportunity"

grid["opportunity_level"] = grid.apply(classify_opportunity, axis=1)

# Rank opportunity areas using opportunity level only.
# High Opportunity -> highest priority
# Medium Opportunity -> next priority
# Low Opportunity -> lowest priority

opportunity_priority = {
    "High Opportunity": 1,
    "Medium Opportunity": 2,
    "Low Opportunity": 3
}

grid["priority_level"] = grid["opportunity_level"].map(opportunity_priority)

grid = grid.sort_values(
    ["priority_level", "total_orders"],
    ascending=[True, False]
).reset_index(drop=True)

grid["priority_rank"] = range(1, len(grid) + 1)

grid["opportunity_id"] = [f"OPP{i:03d}" for i in range(1, len(grid) + 1)]

grid[[
    "opportunity_id", "cluster", "total_orders", "competitors_within_2km",
    "nearest_competitor_km", "opportunity_score", "opportunity_level"
]].head(15)

In [ ]:
# Save the opportunity-area output for the next stage.
output_cols = [
    "opportunity_id", "grid_lat_id", "grid_lon_id", "total_orders", "cluster",
    "area_lat", "area_lon", "competitors_within_2km", "nearest_competitor_km",
    "demand_level", "competition_level", "demand_score",
    "competition_score", "opportunity_score", "opportunity_level", "priority_rank"
]

grid[output_cols].to_csv("../outputs/opportunity_areas.csv", index=False)

print("Saved:", "../outputs/opportunity_areas.csv")
print("\nOpportunity levels:")
print(grid["opportunity_level"].value_counts())

## Interpretation

The output is a shortlist of **potential opportunity areas**. A high opportunity score means the area is attractive under the assumptions used in this case study: strong observed demand and comparatively lower competitor coverage.

This does **not** mean a store should automatically be opened there. Real deployment would require additional data such as road-network travel time, actual company SLA, rider availability, store processing time, and validated business constraints.